# Exploración de Datos — Dashboard de Ventas CR

Análisis exploratorio del dataset de ventas 2024–2025 (5,000 transacciones).
Este notebook documenta los hallazgos que guiaron el diseño del dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0F172A',
    'axes.facecolor':   '#1E293B',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  '#94A3B8',
    'text.color':       '#CBD5E1',
    'xtick.color':      '#64748B',
    'ytick.color':      '#64748B',
    'grid.color':       '#334155',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
})

AZUL   = '#4F8EF7'
TEAL   = '#00C9A7'
AMBER  = '#F59E0B'
ROSA   = '#F472B6'

In [ ]:
df = pd.read_csv('../data/ventas.csv', parse_dates=['fecha'], encoding='utf-8-sig')

print(f'Filas:           {len(df):,}')
print(f'Columnas:        {df.columns.tolist()}')
print(f'Rango de fechas: {df["fecha"].min().date()} a {df["fecha"].max().date()}')
print(f'Valores nulos:\n{df.isnull().sum()}')
df.head()

## 1. Resumen estadístico

In [ ]:
resumen = df[['cantidad', 'precio_unitario', 'total']].describe()
resumen.loc['CV%'] = (resumen.loc['std'] / resumen.loc['mean'] * 100).round(1)
resumen.style.format('{:,.2f}')

## 2. Tendencia mensual de ventas

In [ ]:
ventas_mes = df.groupby(df['fecha'].dt.to_period('M'))['total'].sum()
ventas_mes.index = ventas_mes.index.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(ventas_mes.index, ventas_mes.values, alpha=0.15, color=AZUL)
ax.plot(ventas_mes.index, ventas_mes.values, color=AZUL, linewidth=2)

# Mes pico
idx_max = ventas_mes.idxmax()
ax.scatter(idx_max, ventas_mes[idx_max], color=TEAL, s=80, zorder=5)
ax.annotate(
    f'Pico\n₡{ventas_mes[idx_max]:,.0f}',
    xy=(idx_max, ventas_mes[idx_max]),
    xytext=(15, -30), textcoords='offset points',
    color=TEAL, fontsize=8,
    arrowprops=dict(arrowstyle='->', color=TEAL, lw=1.2)
)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₡{x/1e6:.1f}M'))
ax.set_title('Ventas Mensuales 2024–2025', pad=12, fontsize=13, color='#F1F5F9')
ax.set_xlabel('')
ax.grid(axis='y')
plt.tight_layout()
plt.show()

print(f'Mes con mayores ventas: {idx_max.strftime("%B %Y")} — ₡{ventas_mes[idx_max]:,.0f}')
print(f'Mes con menores ventas: {ventas_mes.idxmin().strftime("%B %Y")} — ₡{ventas_mes.min():,.0f}')
print(f'Crecimiento YoY (2024→2025): {((ventas_mes["2025"].sum() / ventas_mes["2024"].sum()) - 1) * 100:.1f}%')

## 3. Ventas por categoría

In [ ]:
por_cat = df.groupby('categoria')['total'].agg(['sum', 'count', 'mean']).sort_values('sum', ascending=False)
por_cat.columns = ['Total (₡)', 'Transacciones', 'Ticket Promedio (₡)']
por_cat['Participación %'] = (por_cat['Total (₡)'] / por_cat['Total (₡)'].sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colores = [AZUL, TEAL, AMBER, ROSA]

# Barras de total
bars = axes[0].barh(por_cat.index, por_cat['Total (₡)'], color=colores, height=0.5)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₡{x/1e6:.1f}M'))
axes[0].set_title('Total por Categoría', color='#F1F5F9')
axes[0].grid(axis='x')

# Ticket promedio
axes[1].barh(por_cat.index, por_cat['Ticket Promedio (₡)'], color=colores, height=0.5)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₡{x:,.0f}'))
axes[1].set_title('Ticket Promedio por Categoría', color='#F1F5F9')
axes[1].grid(axis='x')

plt.tight_layout()
plt.show()
por_cat.style.format({'Total (₡)': '₡{:,.0f}', 'Ticket Promedio (₡)': '₡{:,.0f}', 'Participación %': '{:.1f}%'})

## 4. Top productos

In [ ]:
top_prod = df.groupby('producto')['total'].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(11, 5))
norm = plt.Normalize(top_prod.min(), top_prod.max())
colores_grad = plt.cm.Blues(norm(top_prod.values))
bars = ax.barh(top_prod.index, top_prod.values, color=colores_grad, height=0.6)

for bar, val in zip(bars, top_prod.values):
    ax.text(val + top_prod.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'₡{val/1e6:.2f}M', va='center', fontsize=8, color='#94A3B8')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₡{x/1e6:.1f}M'))
ax.set_title('Top 10 Productos por Ingresos', color='#F1F5F9', pad=10)
ax.grid(axis='x')
plt.tight_layout()
plt.show()

## 5. Distribución por provincia

In [ ]:
por_region = df.groupby('region').agg(
    total=('total', 'sum'),
    transacciones=('total', 'count'),
    clientes=('cliente_id', 'nunique')
).sort_values('total', ascending=False)

por_region['pct'] = (por_region['total'] / por_region['total'].sum() * 100).round(1)
por_region['ticket_prom'] = por_region['total'] / por_region['transacciones']

print('Ventas por provincia:')
display(por_region.style.format({
    'total': '₡{:,.0f}',
    'ticket_prom': '₡{:,.0f}',
    'pct': '{:.1f}%'
}))

fig, ax = plt.subplots(figsize=(10, 4))
colores_reg = [AZUL, TEAL, AMBER, ROSA, '#A78BFA', '#34D399', '#FB923C']
por_region_asc = por_region.sort_values('total')
ax.barh(por_region_asc.index, por_region_asc['total'], color=colores_reg, height=0.55)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₡{x/1e6:.1f}M'))
ax.set_title('Ventas Totales por Provincia', color='#F1F5F9', pad=10)
ax.grid(axis='x')
plt.tight_layout()
plt.show()

## 6. Comportamiento de clientes

In [ ]:
compras_x_cliente = df.groupby('cliente_id')['total'].agg(['count', 'sum'])
compras_x_cliente.columns = ['num_compras', 'total_gastado']

print(f'Clientes únicos:              {len(compras_x_cliente):,}')
print(f'Compras promedio por cliente: {compras_x_cliente["num_compras"].mean():.1f}')
print(f'Clientes con 1 sola compra:   {(compras_x_cliente["num_compras"] == 1).sum():,} ({(compras_x_cliente["num_compras"] == 1).mean()*100:.1f}%)')
print(f'Clientes con 5+ compras:      {(compras_x_cliente["num_compras"] >= 5).sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(compras_x_cliente['num_compras'], bins=20, color=AZUL, edgecolor='#0F172A', linewidth=0.5)
axes[0].set_title('Distribución de compras por cliente', color='#F1F5F9')
axes[0].set_xlabel('Número de compras')
axes[0].grid(axis='y')

axes[1].hist(compras_x_cliente['total_gastado'], bins=25, color=TEAL, edgecolor='#0F172A', linewidth=0.5)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₡{x/1e3:.0f}K'))
axes[1].set_title('Distribución de gasto total por cliente', color='#F1F5F9')
axes[1].set_xlabel('Gasto total')
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

## 7. Estacionalidad — ventas por día de la semana y mes

In [ ]:
df['dia_semana'] = df['fecha'].dt.day_name()
orden_dias = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
nombres_dias = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']

ventas_dia = df.groupby('dia_semana')['total'].sum().reindex(orden_dias)
ventas_mes_num = df.groupby(df['fecha'].dt.month)['total'].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(nombres_dias, ventas_dia.values, color=AZUL, width=0.6)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₡{x/1e6:.1f}M'))
axes[0].set_title('Ventas por día de la semana', color='#F1F5F9')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y')

meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
axes[1].bar(meses, ventas_mes_num.values, color=TEAL, width=0.6)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₡{x/1e6:.1f}M'))
axes[1].set_title('Ventas por mes (ambos años)', color='#F1F5F9')
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

## 8. Hallazgos clave

| # | Hallazgo | Impacto |
|---|----------|---------|
| 1 | **Software** es la categoría con mayor ingreso (~53% del total) pese a tener menos transacciones — ticket muy alto | Priorizar campañas en este segmento |
| 2 | **San José** concentra el 35% de las ventas, seguido de Alajuela (20%) | Foco de inversión comercial en GAM |
| 3 | Las ventas en **noviembre y diciembre** son ~50% más altas que el promedio mensual | Reforzar inventario y equipo de ventas en Q4 |
| 4 | Los **días de semana** generan el doble que los fines de semana | El canal es principalmente B2B |
| 5 | La mayoría de clientes compra entre 4 y 8 veces en 2 años | Hay oportunidad de loyalty programs |